In [1]:
import sys
import os

# Go up one level to the main project directory and add it to Python's path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

from src.preprocess import preprocess_data

In [4]:
print("Loading and preprocessing training data...")
train_data = pd.read_csv(r'../data/raw/train.csv')
df = preprocess_data(train_data)

X = df.drop('health_condition', axis=1)
y = df['health_condition']

Loading and preprocessing training data...


In [5]:
X_train_local, X_test_local, y_train_local, y_test_local = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
print("Applying SMOTE....")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_local, y_train_local)

local_model = XGBClassifier(
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)
print("Training local model...")
local_model.fit(X_train_smote, y_train_smote)
local_preds = local_model.predict(X_test_local)
macro_f1 = f1_score(y_test_local, local_preds, average='macro')

print(f"\nLocal Validation Macro F1-Score: {macro_f1:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test_local, local_preds))

Applying SMOTE....
Training local model...

Local Validation Macro F1-Score: 0.8215

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.94      0.95    118512
           1       0.76      0.82      0.79      7961
           2       0.65      0.82      0.72     11545

    accuracy                           0.92    138018
   macro avg       0.79      0.86      0.82    138018
weighted avg       0.93      0.92      0.93    138018



Applying SMOTE to ALL training data...


In [10]:
import optuna

def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_extimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'random_state': 42,
        'n_jobs': -1,
        'eval_metric': 'mlogloss'
    }

    model = XGBClassifier(**param)

    model.fit(X_train_smote, y_train_smote)
    preds = model.predict(X_test_local)

    macro_f1 = f1_score(y_test_local, preds, average='macro')
    return macro_f1

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("\n--- Tuning Complete ---")
print(f"Best Macro F1-Score: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

[I 2026-07-29 16:27:46,479] A new study created in memory with name: no-name-a4f81721-f913-433f-91c6-dc4d54050490
[I 2026-07-29 16:28:40,989] Trial 0 finished with value: 0.8026038335808786 and parameters: {'n_extimators': 422, 'learning_rate': 0.021050701589047146, 'max_depth': 7, 'subsample': 0.5361209258301912, 'colsample_bytree': 0.5675616157090928, 'min_child_weight': 1}. Best is trial 0 with value: 0.8026038335808786.
[I 2026-07-29 16:29:04,808] Trial 1 finished with value: 0.8168115589920695 and parameters: {'n_extimators': 219, 'learning_rate': 0.16780439915464396, 'max_depth': 5, 'subsample': 0.6880977524734376, 'colsample_bytree': 0.8180214631724321, 'min_child_weight': 1}. Best is trial 1 with value: 0.8168115589920695.
[I 2026-07-29 16:29:54,135] Trial 2 finished with value: 0.8126020652598108 and parameters: {'n_extimators': 438, 'learning_rate': 0.06834925932506156, 'max_depth': 5, 'subsample': 0.6209818096558275, 'colsample_bytree': 0.9120795828622783, 'min_child_weight'


--- Tuning Complete ---
Best Macro F1-Score: 0.8440
Best Hyperparameters:
    n_extimators: 334
    learning_rate: 0.19109071711581085
    max_depth: 10
    subsample: 0.6702125486428763
    colsample_bytree: 0.8709221679111577
    min_child_weight: 6


In [12]:
print("Applying SMOTE to ALL training data...")
X_full_smote, y_full_smote = smote.fit_resample(X, y)

# 2. Retrain the model on 100% of the SMOTE Data using OPTUNA'S BEST PARAMS
print("Retraining final model with optimized parameters...")

# We unpack **study.best_params to automatically pass in the winning settings
final_model = XGBClassifier(
    **study.best_params, 
    random_state=42, 
    n_jobs=-1, 
    eval_metric='mlogloss'
)

final_model.fit(X_full_smote, y_full_smote)

# 3. Load and preprocess the Kaggle test data
print("Processing Kaggle test data...")
raw_test_df = pd.read_csv(r'../data/raw/test.csv')
passenger_ids = raw_test_df['id']

clean_test_df = preprocess_data(raw_test_df)
X_test_kaggle = clean_test_df.reindex(columns=X.columns, fill_value=0)

# 4. Generate Predictions
print("Generating final predictions...")
kaggle_preds = final_model.predict(X_test_kaggle)

# 5. Format and Save Submission
submission = pd.DataFrame({
    'id': passenger_ids,
    'health_condition': kaggle_preds
})

# Map numeric predictions back to text labels for Kaggle
reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
submission['health_condition'] = submission['health_condition'].map(reverse_mapping)

submission.to_csv('../results/submission_5.csv', index=False)
print("Success! submission.csv is ready for Kaggle upload.")

Applying SMOTE to ALL training data...
Retraining final model with optimized parameters...
Processing Kaggle test data...
Generating final predictions...
Success! submission.csv is ready for Kaggle upload.


In [13]:
from sklearn.utils.class_weight import compute_sample_weight

In [14]:
print("\n--- PHASE 2: Final Predictions ---")
# Retrain on 100% of the Training Data
weights_final = compute_sample_weight(class_weight='balanced', y=y)
final_model = XGBClassifier(
    **study.best_params, 
    random_state=42, 
    n_jobs=-1, 
    eval_metric='mlogloss'
)
print("Retraining model on ALL training data...")

final_model.fit(X, y, sample_weight=weights_final)

print("Loading and preprocessing test data...")
# test_path = os.path.join(data_dir, 'test.csv')
raw_test_df = pd.read_csv(r"..\data\raw\test.csv")
passenger_ids = raw_test_df['id']

# Preprocess test data
clean_test_df = preprocess_data(raw_test_df)

# Align columns perfectly with training data
X_test_kaggle = clean_test_df.reindex(columns=X.columns, fill_value=0)

print("Generating predictions...")
kaggle_preds = final_model.predict(X_test_kaggle)

# Format submission
submission = pd.DataFrame({
    'id': passenger_ids,
    'health_condition': kaggle_preds
})

# Map numeric predictions (0, 1, 2) back to text labels for Kaggle
reverse_mapping = {0: 'at-risk', 1: 'fit', 2: 'unhealthy'}
submission['health_condition'] = submission['health_condition'].map(reverse_mapping)

# Get the next incremental filename and save
save_path = r"../results/submission_6.csv"
submission.to_csv(save_path, index=False)
print(f"\nSuccess! Submission saved at: {save_path}")


--- PHASE 2: Final Predictions ---
Retraining model on ALL training data...
Loading and preprocessing test data...
Generating predictions...

Success! Submission saved at: ../results/submission_6.csv
